# D5 companion — Protein foundation models (ESM-2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/course-curriculum/notebooks/d5-protein-foundation-models.ipynb)

In the chapter we built a *toy* protein embedding from amino-acid composition. Here we compute **real** embeddings with **ESM-2** (`facebook/esm2_t6_8M_UR50D`), a protein language model from Meta AI, and show that similar proteins land close together in embedding space.

> Runs best on Colab with a GPU runtime (*Runtime → Change runtime type → GPU*).

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch

## 2. Check for a GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU found — the 8M model is small enough to run on CPU, '
          'just slower. On Colab: Runtime → Change runtime type → GPU.')

## 3. Load ESM-2

`esm2_t6_8M_UR50D` is the smallest ESM-2 model (about 8 million parameters), trained on millions of protein sequences with masked next-token-style prediction — the protein analogue of the language models in chapter D3.

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = 'facebook/esm2_t6_8M_UR50D'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print(f'Loaded {model_name} with {model.num_parameters():,} parameters.')

## 4. A few short protein sequences

Two pairs that should be close, plus one outlier. Letters are amino acids.

In [ ]:
proteins = {
    'hemoglobin_A':  'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF',
    'hemoglobin_B':  'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFG',
    'insulin':       'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERG',
    'insulin_var':   'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGDRG',
}
for name, seq in proteins.items():
    print(f'{name:>14}: {len(seq)} residues')

## 5. Compute embeddings

We run each sequence through ESM-2 and average its per-residue vectors into a single fixed-length **embedding** — one vector per protein (this is *mean pooling*).

In [ ]:
import numpy as np

def embed(seq):
    inputs = tokenizer(seq, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs).last_hidden_state    # (1, n_tokens, dim)
    return out[0].mean(dim=0).cpu().numpy()        # mean-pool to one vector

embeddings = {name: embed(seq) for name, seq in proteins.items()}
dim = len(next(iter(embeddings.values())))
print(f'Each protein is now a {dim}-dimensional embedding vector.')

## 6. Compare embeddings

We use **cosine similarity** (1 = identical direction). Similar proteins should score high — just like the toy heatmap in the chapter, but from real learned representations.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

names = list(proteins)
V = np.array([embeddings[n] for n in names])
sim = np.array([[cosine(V[i], V[j]) for j in range(len(names))]
                for i in range(len(names))])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap='viridis')
ax.set_xticks(range(len(names)), names, rotation=45, ha='right')
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{sim[i, j]:.2f}', ha='center', va='center', color='white')
ax.set_title('ESM-2 embedding similarity')
fig.colorbar(im, ax=ax, label='cosine similarity')
plt.tight_layout()
plt.show()

The two hemoglobin chains should sit closer to each other, and the two insulin variants closer to each other, than across the groups — ESM-2 has grouped them by **biological similarity** that it learned from sequence alone, just as our toy composition embedding grouped proteins by their ingredients.

## 7. Your turn

- Add your own protein sequence (any string of amino-acid letters) and see where it lands in the similarity matrix.
- Change a single residue in one sequence and check how much the embedding similarity moves — does a small mutation cause a small change?
- Swap in a larger ESM-2 model (e.g. `'facebook/esm2_t12_35M_UR50D'`) and compare the similarities. Do the groupings get sharper?
- Use the per-residue vectors (before mean-pooling) to compare *positions* within a single protein.